In [1]:
import pandas as pd
import glob
import os

# Load all CSV files
csv_files = glob.glob('../data/ver2_*.csv')
print(f"Found {len(csv_files)} files")



Found 5 files


In [2]:
dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    print(f"{os.path.basename(file)}: {len(df)} rows")
    dfs.append(df)

# Combine all
combined = pd.concat(dfs, ignore_index=True)
print(f"\nTotal combined: {len(combined)} sessions")
print(f"Human: {(combined['label']=='human').sum()}")
print(f"Rat: {(combined['label']=='rat').sum()}")

# Save combined
combined.to_csv('ratprobe_combined.csv', index=False)
print("\n✓ Saved: ratprobe_combined.csv")

ver2_anirlen.csv: 30 rows
ver2_batzorig.csv: 33 rows
ver2_esko.csv: 31 rows
ver2_gbdio.csv: 30 rows
ver2_hulan.csv: 35 rows

Total combined: 159 sessions
Human: 109
Rat: 50

✓ Saved: ratprobe_combined.csv


# Study detils

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('ratprobe_combined.csv')

In [6]:
print("="*60)
print("DATASET OVERVIEW")

print("="*60) 
print(f"Total sessions: {len(df)}")
print(f"Human sessions: {(df['label']=='human').sum()}")
print(f"Rat sessions: {(df['label']=='rat').sum()}")
print(f"Features: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")

DATASET OVERVIEW
Total sessions: 159
Human sessions: 109
Rat sessions: 50
Features: 28
Missing values: 0


In [7]:
print("\n" + "="*60)
print("DEVICE DISTRIBUTION")
print("="*60)
print("\nOperating Systems:")
print(df['deviceOs'].value_counts())
print("\nBrowsers:")
print(df['deviceBrowser'].value_counts())


DEVICE DISTRIBUTION

Operating Systems:
deviceOs
Windows    94
macOS      65
Name: count, dtype: int64

Browsers:
deviceBrowser
Chrome     65
Edge       64
Firefox    30
Name: count, dtype: int64


In [8]:
print("\n" + "="*60)
print("NETWORK LATENCY ANALYSIS")
print("="*60)
human_latency = df[df['label']=='human']['networkLatencyMs'].dropna()
rat_latency = df[df['label']=='rat']['networkLatencyMs'].dropna()

print(f"\nHuman latency: mean={human_latency.mean():.1f}ms, std={human_latency.std():.1f}ms")
print(f"Rat latency: mean={rat_latency.mean():.1f}ms, std={rat_latency.std():.1f}ms")
print(f"Ratio: {rat_latency.mean() / human_latency.mean():.1f}x")


NETWORK LATENCY ANALYSIS

Human latency: mean=636.0ms, std=187.8ms
Rat latency: mean=567.0ms, std=81.9ms
Ratio: 0.9x


In [10]:
print("\n" + "="*60)
print("FEATURE STATISTICS")
print("="*60)


features = ['straightnessMean', 'dirEntropyMean', 'velCvMean', 'networkLatencyMs']
for feature in features:
    if feature in df.columns:
        human_vals = df[df['label']=='human'][feature].dropna()
        rat_vals = df[df['label']=='rat'][feature].dropna()
        print(f"\n{feature}:")
        print(f"  Human: {human_vals.mean():.3f} ± {human_vals.std():.3f}")
        print(f"  Rat:   {rat_vals.mean():.3f} ± {rat_vals.std():.3f}")


FEATURE STATISTICS

straightnessMean:
  Human: 0.717 ± 0.151
  Rat:   0.736 ± 0.161

dirEntropyMean:
  Human: 1.739 ± 0.337
  Rat:   1.684 ± 0.379

velCvMean:
  Human: 0.955 ± 0.234
  Rat:   1.661 ± 0.724

networkLatencyMs:
  Human: 636.000 ± 187.764
  Rat:   567.000 ± 81.852


In [11]:
summary = {
    'Total Sessions': len(df),
    'Human Sessions': (df['label']=='human').sum(),
    'Rat Sessions': (df['label']=='rat').sum(),
    'Mean Human Latency (ms)': human_latency.mean(),
    'Mean Rat Latency (ms)': rat_latency.mean(),
}
summary_df = pd.DataFrame([summary])
summary_df.to_csv('data_summary.csv', index=False)
print("\n✓ Saved: data_summary.csv")


✓ Saved: data_summary.csv


In [13]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('ratprobe_combined.csv')

# What actually differs?
print("="*60)
print("LATENCY ANALYSIS")
print("="*60)
h_latency = df[df['label']=='human']['networkLatencyMs'].dropna()
r_latency = df[df['label']=='rat']['networkLatencyMs'].dropna()

print(f"Human latency: {h_latency.mean():.0f}ms ± {h_latency.std():.0f}ms")
print(f"Remote latency: {r_latency.mean():.0f}ms ± {r_latency.std():.0f}ms")
print(f"Difference: {h_latency.mean() - r_latency.mean():.0f}ms")

# Statistical test
t_stat, p_val = stats.ttest_ind(h_latency, r_latency)
print(f"P-value: {p_val:.6f}")

print("\n" + "="*60)
print("BEHAVIORAL ANALYSIS")
print("="*60)

# Check behavioral differences
for feature in ['straightnessMean', 'dirEntropyMean', 'velCvMean']:
    h = df[df['label']=='human'][feature].dropna()
    r = df[df['label']=='rat'][feature].dropna()
    
    t_stat, p_val = stats.ttest_ind(h, r)
    print(f"{feature}:")
    print(f"  Human: {h.mean():.3f} ± {h.std():.3f}")
    print(f"  RAT: {r.mean():.3f} ± {r.std():.3f}")
    print(f"  P-value: {p_val:.6f} {'***' if p_val < 0.001 else 'ns'}")
    print()

print("Note: If behavioral p-values are high (>0.05), behavior doesn't differ!")
print("This is EXPECTED because both human and RAT are controlled by humans!")

LATENCY ANALYSIS
Human latency: 636ms ± 188ms
Remote latency: 567ms ± 82ms
Difference: 69ms
P-value: 0.013855

BEHAVIORAL ANALYSIS
straightnessMean:
  Human: 0.717 ± 0.151
  RAT: 0.736 ± 0.161
  P-value: 0.460711 ns

dirEntropyMean:
  Human: 1.739 ± 0.337
  RAT: 1.684 ± 0.379
  P-value: 0.354427 ns

velCvMean:
  Human: 0.955 ± 0.234
  RAT: 1.661 ± 0.724
  P-value: 0.000000 ***

Note: If behavioral p-values are high (>0.05), behavior doesn't differ!
This is EXPECTED because both human and RAT are controlled by humans!


In [ ]:
"""
Cohen's d 和显著性水平计算示例代码
用于复现上述统计结果
"""

import pandas as pd
import numpy as np
from scipy import stats

def cohens_d(group1, group2):
    """计算Cohen's d效应量（合并标准差版本）"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    
    # 合并标准差
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    
    # Cohen's d = (M2 - M1) / pooled_std
    d = (group2.mean() - group1.mean()) / pooled_std
    return d

def significance_stars(p_value):
    """根据p值返回显著性符号"""
    if p_value < 0.001:
        return "★★★"
    elif p_value < 0.01:
        return "★★"
    elif p_value < 0.05:
        return "★"
    else:
        return "ns"

def effect_size_interpretation(d):
    """解读效应量大小"""
    d_abs = abs(d)
    if d_abs < 0.2:
        return "可忽略"
    elif d_abs < 0.5:
        return "小效应"
    elif d_abs < 0.8:
        return "中等效应"
    else:
        return "大效应"



In [4]:
df = pd.read_csv('ratprobe_combined.csv')

human = df[df['label'] == 'human']['velCvMean']
rat = df[df['label'] == 'rat']['velCvMean']

t_stat, p_val = stats.ttest_ind(human, rat, equal_var=False)  # Welch's t-test
d = cohens_d(human, rat)

print(f"t = {t_stat:.3f}, p = {p_val:.6f}, d = {d:.3f}")
print(f"显著性: {significance_stars(p_val)}, 效应量: {effect_size_interpretation(d)}")

t = -6.726, p = 0.000000, d = 1.571
显著性: ★★★, 效应量: 大效应


In [ ]:
human = df[df['label'] == 'human']['accelStdMean']
rat = df[df['label'] == 'rat']['accelStdMean']

t_stat, p_val = stats.ttest_ind(human, rat, equal_var=False)  # Welch's t-test
d = cohens_d(human, rat)

print(f"t = {t_stat:.3f}, p = {p_val:.6f}, d = {d:.3f}")
print(f"显著性: {significance_stars(p_val)}, 效应量: {effect_size_interpretation(d)}")

t = -3.082, p = 0.003371, d = 0.780
显著性: ★★, 效应量: 中等效应
